# Data Analysis & Feature Engineering

Goal: understand what makes a loop benefit from unrolling and create better features

Current accuracy: 92.3% but might be too easy (88.7% positive class)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

project_root = Path.cwd().parent

## Load Data

In [ ]:
df = pd.read_csv(project_root / 'data' / 'processed' / 'dataset.csv')
print(f"Dataset: {len(df)} loops from {df['source_file'].nunique()} programs")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# basic stats
print("Class distribution:")
print(df['beneficial'].value_counts())
print(f"\nBeneficial: {df['beneficial'].mean()*100:.1f}%")
print(f"\nSpeedup stats:")
print(df['speedup'].describe())

## Class Imbalance Problem

88.7% beneficial means a dumb classifier that always says "beneficial" gets 88.7% accuracy.

Our 92.3% isn't that impressive relative to baseline.

In [ ]:
# speedup distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(df['speedup'], bins=30, edgecolor='black', alpha=0.7)
ax1.axvline(1.0, color='red', linestyle='--', label='No change')
ax1.axvline(1.05, color='orange', linestyle='--', label='Threshold (1.05x)')
ax1.set_xlabel('Speedup')
ax1.set_ylabel('Count')
ax1.set_title('Speedup Distribution')
ax1.legend()

# by class
beneficial = df[df['beneficial'] == 1]['speedup']
not_beneficial = df[df['beneficial'] == 0]['speedup']

ax2.hist([beneficial, not_beneficial], bins=20, label=['Beneficial', 'Not beneficial'], alpha=0.7)
ax2.axvline(1.05, color='orange', linestyle='--')
ax2.set_xlabel('Speedup')
ax2.set_ylabel('Count')
ax2.set_title('Speedup by Class')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Beneficial range: {beneficial.min():.3f} - {beneficial.max():.3f}")
print(f"Not beneficial range: {not_beneficial.min():.3f} - {not_beneficial.max():.3f}")

## Feature Analysis

In [ ]:
# current features
feature_cols = [
    'num_instructions', 'num_basic_blocks', 'num_load_instructions',
    'num_store_instructions', 'num_branches', 'num_calls', 'num_arithmetic_ops',
    'estimated_trip_count', 'has_constant_trip_count', 'nesting_depth',
    'num_phi_nodes', 'num_memory_dependencies', 'has_early_exit', 'num_exits'
]

df[feature_cols].describe()

In [ ]:
# check for low variance (features that don't vary much)
print("Feature variance:")
for col in feature_cols:
    unique_vals = df[col].nunique()
    print(f"{col}: {unique_vals} unique values")
    if unique_vals <= 3:
        print(f"  Values: {df[col].unique()}")

## Problem: Most features are constant!

If most loops have the same values, the model can't learn from them.

In [ ]:
# correlations with target
correlations = df[feature_cols + ['beneficial']].corr()['beneficial'].drop('beneficial').sort_values()

plt.figure(figsize=(10, 6))
correlations.plot(kind='barh', color=correlations.apply(lambda x: 'green' if x > 0 else 'red'), alpha=0.7)
plt.xlabel('Correlation with Beneficial')
plt.title('Feature Correlations')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nStrongest positive correlations:")
print(correlations.tail(3))
print("\nStrongest negative correlations:")
print(correlations.head(3))

In [ ]:
# group by program to see patterns
program_stats = df.groupby('source_file').agg({
    'beneficial': ['sum', 'count', 'mean'],
    'speedup': ['mean', 'min', 'max'],
    'num_instructions': 'mean'
}).round(3)

program_stats.columns = ['_'.join(col) for col in program_stats.columns]
program_stats = program_stats.sort_values('beneficial_mean', ascending=False)
print(program_stats)

## Feature Engineering

Create derived features that might be more informative:
- Ratios (memory intensity, arithmetic intensity)
- Interactions
- Polynomial features

In [ ]:
# create new features
df_eng = df.copy()

# memory intensity
df_eng['memory_ops'] = df_eng['num_load_instructions'] + df_eng['num_store_instructions']
df_eng['memory_ratio'] = df_eng['memory_ops'] / (df_eng['num_instructions'] + 1)  # +1 to avoid div by 0

# compute intensity  
df_eng['compute_ratio'] = df_eng['num_arithmetic_ops'] / (df_eng['num_instructions'] + 1)

# control flow complexity
df_eng['branch_ratio'] = df_eng['num_branches'] / (df_eng['num_instructions'] + 1)
df_eng['phi_ratio'] = df_eng['num_phi_nodes'] / (df_eng['num_instructions'] + 1)

# loop size categories (instead of raw trip count)
df_eng['trip_count_log'] = np.log10(df_eng['estimated_trip_count'].replace(-1, 1000000) + 1)

# instruction density
df_eng['instructions_per_bb'] = df_eng['num_instructions'] / (df_eng['num_basic_blocks'] + 1)

# call overhead
df_eng['has_calls'] = (df_eng['num_calls'] > 0).astype(int)

# memory dependencies per memory op
df_eng['deps_per_memop'] = df_eng['num_memory_dependencies'] / (df_eng['memory_ops'] + 1)

print("New features created:")
new_features = ['memory_ratio', 'compute_ratio', 'branch_ratio', 'phi_ratio', 
                'trip_count_log', 'instructions_per_bb', 'has_calls', 'deps_per_memop']
print(df_eng[new_features].describe())

In [ ]:
# check correlations of new features
new_corr = df_eng[new_features + ['beneficial']].corr()['beneficial'].drop('beneficial').sort_values()

plt.figure(figsize=(10, 5))
new_corr.plot(kind='barh', color=new_corr.apply(lambda x: 'green' if x > 0 else 'red'), alpha=0.7)
plt.xlabel('Correlation with Beneficial')
plt.title('New Feature Correlations')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(new_corr)

## Visualize Relationships

In [ ]:
# scatter plots of new features vs speedup
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

scatter_features = ['memory_ratio', 'compute_ratio', 'trip_count_log', 'deps_per_memop']

for idx, feat in enumerate(scatter_features):
    ax = axes[idx]
    colors = df_eng['beneficial'].map({0: 'red', 1: 'green'})
    ax.scatter(df_eng[feat], df_eng['speedup'], c=colors, alpha=0.6)
    ax.set_xlabel(feat)
    ax.set_ylabel('Speedup')
    ax.set_title(f'{feat} vs Speedup')
    ax.axhline(1.05, color='orange', linestyle='--', alpha=0.5)
    
    # add correlation
    corr = df_eng[[feat, 'speedup']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'corr={corr:.3f}', transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Train with New Features

In [ ]:
# combine old and new features
all_features = feature_cols + new_features

X = df_eng[all_features].copy()
y = df_eng['beneficial'].copy()

# handle -1 trip counts
X['estimated_trip_count'] = X['estimated_trip_count'].replace(-1, 1000000)

# fill any NaN from division
X = X.fillna(0)

print(f"Features: {len(all_features)}")
print(f"Samples: {len(X)}")

In [ ]:
# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# train models with new features
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10),
}

results = {}

print("="*70)
print("Model Performance with Engineered Features")
print("="*70)

for name, model in models.items():
    print(f"\n{name}:")
    
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print(f"  Accuracy:  {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall:    {rec:.3f}")
    print(f"  F1-Score:  {f1:.3f}")
    
    results[name] = {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'model': model}
    
    # feature importance
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        top_features = sorted(zip(all_features, importances), key=lambda x: -x[1])[:5]
        print(f"  Top 5 features:")
        for feat, imp in top_features:
            print(f"    {feat}: {imp:.3f}")

## Feature Importance Deep Dive

In [ ]:
# random forest feature importances
rf_model = results['Random Forest']['model']
importances = rf_model.feature_importances_
feature_imp = pd.DataFrame({
    'feature': all_features,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(feature_imp['feature'][:15], feature_imp['importance'][:15])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(feature_imp.head(10))

## Confusion Matrix

In [ ]:
# confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, result) in enumerate(results.items()):
    model = result['model']
    
    if 'Logistic' in name:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Not Beneficial', 'Beneficial'],
                yticklabels=['Not Beneficial', 'Beneficial'])
    axes[idx].set_title(f'{name}\n(Acc: {result["acc"]:.3f})')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## Error Analysis

Look at misclassified examples

In [ ]:
# use random forest predictions
rf_model = results['Random Forest']['model']
y_pred = rf_model.predict(X_test)

# find errors
errors = X_test[y_test != y_pred]
error_labels = y_test[y_test != y_pred]
error_preds = y_pred[y_test != y_pred]

print(f"Misclassified: {len(errors)} / {len(X_test)}")

if len(errors) > 0:
    error_df = df_eng.iloc[errors.index][['source_file', 'speedup', 'beneficial'] + new_features]
    print("\nMisclassified loops:")
    print(error_df)

## Summary & Insights

Key findings:
1. Dataset is imbalanced (88.7% positive)
2. Many original features have low variance (constant across loops)
3. Engineered ratio features might be more informative
4. Small dataset (62 samples) limits what we can learn

Next steps:
- Compare against LLVM's actual decisions
- Collect more data from real benchmarks
- Try regression (predict speedup directly) instead of classification
- Experiment with threshold (currently 1.05x)

In [ ]:
# save best model
import pickle

best_model = results['Random Forest']['model']
model_dir = project_root / 'models'
model_dir.mkdir(exist_ok=True)

with open(model_dir / 'rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
    
with open(model_dir / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# save feature names for prediction
with open(model_dir / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(all_features))

print("✓ Model saved")
print(f"  Random Forest accuracy: {results['Random Forest']['acc']:.3f}")
print(f"  Features: {len(all_features)}")